In [1]:
import logging
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import pandas as pd
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
logging.basicConfig(level=logging.INFO)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:

def generate_base_scorecard_from_csv(file_path):
    # Load the data
    df = pd.read_csv(file_path)

    # 1. Clean column names
    # This removes 'eval_metrics/', '[0.5]', '[512]', '[mean]', and '[0.1-0.9]'
    def clean_header(col):
        if '/' in col:
            col = col.split('/')[-1]
        # Remove anything inside brackets
        import re
        col = re.sub(r'\[.*?\]', '', col)
        return col.strip()

    df.columns = [clean_header(c) for c in df.columns]

    # 2. Define the metric grouping
    metric_groups = {
        "Accuracy": [
            "MSE", "RMSE", "MAE", "MASE", "MAPE", "sMAPE", 
            "NRMSE", "ND", "mean_weighted_sum_quantile_loss"
        ],
        "Fidelity & Dynamics": [
            "MeanAttractionRatio", 
            "PathVolatilityRatio", 
            "PersistenceSkillRatio", 
            "MeanSkillRatio",        
            "RelativeUncertainty"    
        ],
        "Bias": [
            "MeanBiasRatio", 
            "NormalizedMeanBias",    
            "MSIS"
        ]
    }

    # 3. Create a MultiIndex DataFrame for better organization
    id_cols = ['dataset', 'num_variates', 'model']
    
    # Flatten metrics to ensure we only include what exists in the file
    available_metrics = [m for sublist in metric_groups.values() for m in sublist if m in df.columns]
    
    # Pivot the data
    scorecard = df[id_cols + available_metrics].set_index('dataset')

    # 4. Final Formatting (Optional: Transpose for a vertical scorecard)
    vertical_scorecard = scorecard.transpose()

    return scorecard

In [10]:
import pandas as pd
import re

# copied from metrics.py
"""
Defining some new metrics to measure biases and characteristics of timeseries forecasts

| Metric | Category | Ideal Value | What it detects |
| :--- | :--- | :--- | :--- |
| Persistence Skill Ratio (PSR) | Skill (Accuracy) | < 1.0 | Is the model better than a "naive" flat line starting from the last known value? |
| Mean Skill Ratio (MSR) | Skill (Accuracy) | < 1.0 | Is the model better than just guessing the historical average of the series? |
| Mean Attraction Ratio (MAR) | Dynamics (Energy) | 1.0 | Does the forecast match the global amplitude (distance from the mean) of the truth? |
| Path Volatility Ratio (PVR) | Dynamics (Energy) | 1.0 | Does the path have the correct "wiggliness" and transition from history? |
| Mean Bias Ratio (MBR) | Dynamics (Frequency) | 0.5 | How often does the model play it safe by landing between the mean and the target? |
| Normalized Mean Bias (NMB) | Direction | 0.0 | Is the model systematically "Bearish" (under-predicting) or "Bullish" (over-predicting)? |
| Relative Uncertainty (RU) | Sharpness | ~0.0* | How much "doubt" (interval width) does the model express relative to signal scale? |

*Ideal RU depends on data noise; in deterministic signals, it should be near 0.

Possible model behaviors:
- The "Lazy" Model: High PSR (>1) and high MSR (>1). The model is literally worse than a flat line.
- The "Muted" Model: MAR > 1.2 and PVR > 1.2. The model gets the direction right but significantly under-sizes the moves.
- The "Hallucinator": PVR < 0.8. The model is adding "jitter" or noise that isn't present in the ground truth.
- The "Hedger": MBR > 0.7. The model is systematically "leaning" toward the mean to minimize MSE loss.
- The "Terrified" Model: High RU (> 0.2) on simple patterns. The model is inflating its uncertainty intervals.
"""

# 1. STYLE PALETTE: Centralized visual theme
PALETTE = {
    "good": "background-color: #c3e6cb; color: #155724;",           # Green
    "bad":  "background-color: #f8d7da; color: #721c24; font-weight: bold;", # Red
    "warning": "background-color: #fff3cd; color: #856404;",          # Yellow
    "neutral": "background-color: #e2e3e5; color: #383d41;",        # Grey
    "sharp": "background-color: #fce4ec; color: #880e4f;"           # Pink
}

# 2. THRESHOLDS: Centralized business logic
METRIC_THRESHOLDS = {
    "PersistenceSkillRatio": {"good": 1.0, "bad": 1.05},
    "MeanSkillRatio":        {"good": 1.0, "bad": 1.05},
    "MeanAttractionRatio":   {"low_good": 0.9, "high_good": 1.1, "bad_high": 1.2, "bad_low": 0.8},
    "PathVolatilityRatio":   {"low_good": 0.9, "high_good": 1.1, "bad_high": 1.2, "bad_low": 0.8},
    "MeanBiasRatio":         {"low_good": 0.45, "high_good": 0.55, "bad_high": 0.6, "bad_low": 0.4},
    "NormalizedMeanBias":    {"good_abs": 0.02, "bad_abs": 0.1},
    "RelativeUncertainty":   {"good": 0.05, "bad": 0.1}
}

def generate_scorecard_from_csv(file_path, output_report_path=None, distributions=None, dataset_types=None, metrics=None):
    
    id_cols = ['dataset_slug', 'num_variates', 'model']

    df = pd.read_csv(file_path)

    # Collision-Aware Cleaning to prevent Styler KeyError
    def clean_header_list(columns):
        cleaned = []
        raw_names = [c.split('/')[-1] if '/' in c else c for c in columns]
        for name in raw_names:
            simple_name = re.sub(r'\[.*?\]', '', name).strip()
            if sum(1 for n in raw_names if re.sub(r'\[.*?\]', '', n).strip() == simple_name) > 1:
                cleaned.append(name.strip())
            else:
                cleaned.append(simple_name)
        return cleaned

    df.columns = clean_header_list(df.columns)
    if metrics:
        df = df[id_cols+metrics]
        
    df = df.drop_duplicates(subset=['dataset_slug', 'model'], keep='last')
    
    if distributions:
        pattern = f"_{'__|_'.join(distributions)}__"
        if "constant" in distributions:
            df = df[~df['dataset_slug'].str.contains('piecewise_constant', na=False)]
        df = df[df['dataset_slug'].str.contains(pattern, na=False)]

    if dataset_type:
        df = df[df['dataset_slug'].str.contains(f"{'|'.join(dataset_types)}", na=False)]

    accuracy_metrics = [c for c in df.columns if any(m in c for m in ["MSE", "RMSE", "MAE", "MAPE", "ND"])]
    dynamics_metrics = [c for c in df.columns if any(k in c for k in METRIC_THRESHOLDS.keys())]
    
    all_selected = list(dict.fromkeys(id_cols + accuracy_metrics + dynamics_metrics))
    
    # RECONSTRUCTION: Ensure unique index and columns
    scorecard = df[all_selected].copy().reset_index(drop=True)
    scorecard.columns = [str(c) for c in scorecard.columns]

    return scorecard

def highlight_behaviours(scorecard):
    id_cols = ['dataset_slug', 'num_variates', 'model']

    # 3. REFACTORED STYLING: Clean logic using Palette and Thresholds
    def highlight_behaviors(s):
        name = s.name
        styles = [""] * len(s)
        
        key = next((k for k in METRIC_THRESHOLDS.keys() if k in name), None)
        if not key: return styles
        
        t = METRIC_THRESHOLDS[key]
        for i, val in enumerate(s):
            styles[i] = PALETTE["neutral"]

            if pd.isna(val) or isinstance(val, str): continue

            # Skill Metrics
            if key in ["PersistenceSkillRatio", "MeanSkillRatio"]:
                if val <= t["good"]: styles[i] = PALETTE["good"]
                elif t["good"] < val < t["bad"]: styles[i] = PALETTE["warning"]
                elif val > t["bad"]: styles[i] = PALETTE["bad"]
            
            # Dynamics (MAR/PVR)
            elif key in ["MeanAttractionRatio", "PathVolatilityRatio"]:
                if t["low_good"] <= val <= t["high_good"]: styles[i] = PALETTE["good"]
                elif t["bad_low"] <= val <= t["bad_high"]: styles[i] = PALETTE["warning"]
                else: styles[i] = PALETTE["bad"]
            
            # Bias (MBR)
            elif key == "MeanBiasRatio":
                if t["low_good"] <= val <= t["high_good"]: styles[i] = PALETTE["good"]
                elif t["bad_low"] <= val <= t["bad_high"]: styles[i] = PALETTE["warning"]
                else: styles[i] = PALETTE["bad"]
            

            # Bias (NMB)
            elif key == "NormalizedMeanBias":
                if abs(val) < t["good_abs"]: styles[i] = PALETTE["good"]
                elif abs(val) < t["bad_abs"]: styles[i] = PALETTE["warning"]
                else: styles[i] = PALETTE["bad"]
            
            # Sharpness (RU)
            elif key == "RelativeUncertainty":
                if val < t["good"]: styles[i] = PALETTE["good"]
                elif t["good"] < val < t["bad"]: styles[i] = PALETTE["warning"]
                else: styles[i] = PALETTE["bad"]
                
        return styles

    # Formatting logic
    def get_format_map(df, metrics):
        f_map = {}
        for m in metrics:
            if pd.api.types.is_numeric_dtype(df[m]) and not df[m].isnull().all():
                if df[m].max() > 1000 or (df[m].min() < 0.001 and df[m].min() > 0):
                    f_map[m] = "{:.4e}"
                else: f_map[m] = "{:.4f}"
            else: f_map[m] = "{}"
        return f_map

    final_metrics = [c for c in scorecard.columns if c not in id_cols]
    
    return (
        scorecard.style
        .apply(highlight_behaviors, axis=0, subset=final_metrics)
        .format(get_format_map(scorecard, final_metrics))
        .hide(axis="index")
    )

In [19]:

results_repertoire_raw = {
    # "ToTo": "../results/simpletime_short_toto_20260208-093620/SimpleTime-Toto/combined_results.csv",
    "Chronos2": "../results_shared/simpletime/simpletime_short_chronos2_20260212-231925/SimpleTime-Chronos2/combined_results.csv",
    "Toto": "../results_shared/simpletime/simpletime_short_toto_20260212-231908/SimpleTime-Toto/combined_results.csv",
    "Moirai": "../results_shared/simpletime/simpletime_short_moirai_20260212-233325/SimpleTime-Moirai/combined_results.csv",
}


# 'metrics': ['MeanAttractionRatio', 'MeanBiasRatio',	'PathVolatilityRatio', 'PersistenceSkillRatio', 'MeanSkillRatio', 'NormalizedMeanBias', 'RelativeUncertainty']


regimes = {
    'Deterministic Trends': {
        'distributions': ['constant', 'linear', 'staircase', 'power_law', 'exponential'],
        'dataset_types': ['univariate'],
        'metrics': ['MeanAttractionRatio', 'MeanBiasRatio',	'PathVolatilityRatio', 'NormalizedMeanBias', 'RelativeUncertainty']
    },
    'Harmonic & Periodic': {
        'distributions': ['sine', 'sawtooth', 'square', 'triangle', 'noise_patch', 'fourier'],
        'dataset_types': ['univariate'],        
        'metrics': ['MeanAttractionRatio', 'MeanBiasRatio',	'PathVolatilityRatio', 'NormalizedMeanBias', 'RelativeUncertainty']
    },
    'Transient Dynamics': {
        'distributions': ['damped_sine', 'growing_sine', 'chirp'],
        'dataset_types': ['univariate'], 
        'metrics': ['MeanAttractionRatio', 'MeanBiasRatio',	'PathVolatilityRatio', 'NormalizedMeanBias', 'RelativeUncertainty']
    },
    # 'Autocorrelated Processes': {
    #     'distributions': ['ar', 'ma', 'arma', 'sarima', 'garch', 'pink_noise', 'fractional_brownian', 'markov_chain'],
    #     'dataset_types': ['univariate']
    # },
    'I.I.D. White Noise': {
        'distributions': ['normal', 'uniform', 'poisson', 'binary', 'student_t', 'lognormal', 'laplace', 'cauchy', 'skew_normal'],
        'dataset_types': ['univariate', 'multivariate', 'multivariate_lagged'],
        'metrics': ['MeanSkillRatio']
    },
    'Stochastic Drift/Shocks': {
        'distributions': ['random_walk', 'piecewise_constant', 'gbm', 'intermittent', 'impulse', 'logistic'],
        'dataset_types': ['univariate', 'multivariate', 'multivariate_lagged'],
        'metrics': ['PersistenceSkillRatio']
    }
}

for regime in regimes.keys():
    distributions = regimes[regime]['distributions']
    dataset_types = regimes[regime]['dataset_types']
    metrics = regimes[regime]['metrics']
    print(distributions, dataset_types)
    
    scorecards = []
    for model in results_repertoire_raw.keys():
        scorecard = generate_scorecard_from_csv(
            results_repertoire_raw[model], "test-scorecard.html",
            distributions=distributions, dataset_types=dataset_types, metrics=metrics
            )
    
        scorecards.append(scorecard)

    final_scorecard = pd.concat(scorecards, keys=results_repertoire_raw.keys(), ignore_index=False)
    final_scorecard = highlight_behaviours(final_scorecard)
    display(final_scorecard)

['constant', 'linear', 'staircase', 'power_law', 'exponential'] ['univariate']


dataset_slug,num_variates,model,MeanAttractionRatio,MeanBiasRatio,PathVolatilityRatio,NormalizedMeanBias,RelativeUncertainty
univariate_constant__short,1,SimpleTime-Chronos2,9.9697e-01,0.3070,0.0000,-0.0000,1.1388e-08
univariate_linear__short,1,SimpleTime-Chronos2,1.0066e+00,0.9789,1.0257,0.0002,3.7900e-02
univariate_staircase__short,1,SimpleTime-Chronos2,1.0069e+00,0.7223,0.7038,0.0008,6.3498e-02
univariate_power_law__short,1,SimpleTime-Chronos2,1.0168e+00,0.7979,1.0694,0.0133,1.0068e-01
univariate_exponential__short,1,SimpleTime-Chronos2,1.3149e+00,0.9157,1.9010,0.2250,2.4119e-01
univariate_constant__short,1,SimpleTime-Toto,6.0759e-05,0.3324,0.0000,-0.0002,7.8289e-03
univariate_linear__short,1,SimpleTime-Toto,9.9732e-01,0.1212,0.9921,-0.0008,7.8566e-02
univariate_staircase__short,1,SimpleTime-Toto,1.0365e+00,0.6265,0.5744,-0.0245,1.9781e-01
univariate_power_law__short,1,SimpleTime-Toto,1.0133e+00,0.9024,1.0597,0.0106,3.9677e-02
univariate_exponential__short,1,SimpleTime-Toto,1.5789e+00,0.9887,1.8776,0.3445,3.6248e-01


['sine', 'sawtooth', 'square', 'triangle', 'noise_patch', 'fourier'] ['univariate']


dataset_slug,num_variates,model,MeanAttractionRatio,MeanBiasRatio,PathVolatilityRatio,NormalizedMeanBias,RelativeUncertainty
univariate_sine__short,1,SimpleTime-Chronos2,1.0083,0.6283,1.0148,-0.0008,0.1351
univariate_sawtooth__short,1,SimpleTime-Chronos2,1.0024,0.5575,0.9951,-0.0021,0.1578
univariate_square__short,1,SimpleTime-Chronos2,1.0133,0.5435,0.9872,-0.0052,0.1320
univariate_triangle__short,1,SimpleTime-Chronos2,1.0031,0.4700,1.0550,0.0003,0.1847
univariate_noise_patch__short,1,SimpleTime-Chronos2,0.9928,0.5004,0.9943,-0.0012,0.1773
univariate_fourier__short,1,SimpleTime-Chronos2,1.0088,0.5473,1.0093,-0.0019,0.3169
univariate_damped_sine__short,1,SimpleTime-Chronos2,1.0844,0.6759,1.0986,-0.0118,0.5607
univariate_growing_sine__short,1,SimpleTime-Chronos2,1.1723,0.8960,1.1811,0.0083,0.3872
univariate_sine__short,1,SimpleTime-Toto,0.9989,0.5050,1.0436,0.0256,0.7215
univariate_sawtooth__short,1,SimpleTime-Toto,1.0083,0.5317,1.0439,-0.0004,0.5380


['damped_sine', 'growing_sine', 'chirp'] ['univariate']


dataset_slug,num_variates,model,MeanAttractionRatio,MeanBiasRatio,PathVolatilityRatio,NormalizedMeanBias,RelativeUncertainty
univariate_damped_sine__short,1,SimpleTime-Chronos2,1.0844,0.6759,1.0986,-0.0118,0.5607
univariate_growing_sine__short,1,SimpleTime-Chronos2,1.1723,0.8960,1.1811,0.0083,0.3872
univariate_chirp__short,1,SimpleTime-Chronos2,1.9658,0.8670,2.8333,-0.0270,2.6859
univariate_damped_sine__short,1,SimpleTime-Toto,1.0271,0.5507,1.0877,0.0666,4.5499
univariate_growing_sine__short,1,SimpleTime-Toto,2.0059,0.8944,2.0837,0.0149,0.7096
univariate_chirp__short,1,SimpleTime-Toto,2.0957,0.8768,2.6777,0.0206,2.6174
univariate_damped_sine__short,1,SimpleTime-Moirai,1.1918,0.7831,1.2797,-0.0982,10.4618
univariate_growing_sine__short,1,SimpleTime-Moirai,4.2795,0.9294,6.2951,0.0062,1.3652
univariate_chirp__short,1,SimpleTime-Moirai,2.6937,0.9153,3.2302,0.0880,3.0721


['normal', 'uniform', 'poisson', 'binary', 'student_t', 'lognormal', 'laplace', 'cauchy', 'skew_normal'] ['univariate', 'multivariate', 'multivariate_lagged']


dataset_slug,num_variates,model,MeanSkillRatio
univariate_normal__short,1,SimpleTime-Chronos2,1.0037
multivariate_normal__short,2,SimpleTime-Chronos2,1.0020
multivariate_lagged_normal__short,2,SimpleTime-Chronos2,1.0001
univariate_uniform__short,1,SimpleTime-Chronos2,1.0042
multivariate_uniform__short,2,SimpleTime-Chronos2,1.0010
multivariate_lagged_uniform__short,2,SimpleTime-Chronos2,0.9962
univariate_poisson__short,1,SimpleTime-Chronos2,0.9786
multivariate_poisson__short,2,SimpleTime-Chronos2,0.9765
multivariate_lagged_poisson__short,2,SimpleTime-Chronos2,0.9745
univariate_binary__short,1,SimpleTime-Chronos2,0.7777


['random_walk', 'piecewise_constant', 'gbm', 'intermittent', 'impulse', 'logistic'] ['univariate', 'multivariate', 'multivariate_lagged']


dataset_slug,num_variates,model,PersistenceSkillRatio
univariate_random_walk__short,1,SimpleTime-Chronos2,1.0167
multivariate_random_walk__short,2,SimpleTime-Chronos2,1.0162
multivariate_lagged_random_walk__short,2,SimpleTime-Chronos2,0.9844
univariate_piecewise_constant__short,1,SimpleTime-Chronos2,1.1360
multivariate_piecewise_constant__short,2,SimpleTime-Chronos2,1.1188
multivariate_lagged_piecewise_constant__short,2,SimpleTime-Chronos2,1.0843
univariate_gbm__short,1,SimpleTime-Chronos2,0.9384
multivariate_gbm__short,2,SimpleTime-Chronos2,0.9540
multivariate_lagged_gbm__short,2,SimpleTime-Chronos2,0.9418
univariate_intermittent__short,1,SimpleTime-Chronos2,0.5787
